### **Block 1: Setup & Sentiment Calibration**
This block initializes the core libraries, sets up the translation engine, and performs **Strategic Sentiment Calibration**. We overwrite the default VADER lexicon with custom weights for:

**Singlish Slang:** Ensuring local sentiment (e.g., "shiok", "jialat") is captured correctly within the Singaporean context.

**Reputational Keywords:** Tailoring scores for monitoring (e.g., "accountable", "scandal", "bloodbank") to ensure organizational health and crisis indicators are weighted accurately. <br>
<br>


**Key Components:** <br>

**VADER Sentiment Analyzer:** The primary engine used for calculating emotional polarity.

**Google Translator API:** Utilized to translate Chinese (ZH), Malay (MS), and Tamil (TA) media back into English for standardized scoring.

**Lexicon Overrides:** Manually injecting positive (1.8), negative (-1.8), and neutral (0.0) values into the model to bypass default settings that might misinterpret humanitarian or local slang terms.


In [2]:
import pandas as pd
import json
import traceback
from datetime import date
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from deep_translator import GoogleTranslator

# --- BLOCK 1: SETUP & VALIDATION ---

try:
    # Initialize tools
    analyzer = SentimentIntensityAnalyzer()
    translator = GoogleTranslator(source='auto', target='en')

    # --- SENTIMENT ALLOCATION & MASTER LEXICON ---
    positive_score, negative_score, neutral_score = 1.8, -1.8, 0.0
    final_lexicon = {}

    # 1. Slang & Local Overrides
    full_slang = {
    # --- POSITIVE & PRAISE (High Scores) ---
    "shiok": 4.0, "steady": 3.0, "steady poon pi pi": 4.5, "zai": 3.0, "ai zai": 3.0,
    "swee": 2.5, "ups": 2.5, "power": 3.0, "tok kong": 3.5, "kilat": 3.5, 
    "solid": 2.5, "huat": 2.5, "ho say": 2.0, "relak": 1.5, "on the ball": 2.5,
    "chop-chop": 2.0, "on": 1.5, "shining": 2.0, "win liao": 2.5, "sedap": 2.0,

    # --- NEGATIVE & FRUSTRATION (Low Scores) ---
    "jialat": -3.5, "jia lat": -3.5, "sian": -3.0, "see peh sian": -4.0, "rabak": -3.5, 
    "sabo": -3.0, "sabo king": -3.5, "leceh": -3.0, "pek chek": -3.5, "tu lan": -4.0, 
    "suay": -3.0, "koyak": -2.5, "teruk": -2.5, "buay tahan": -3.5, "vomit blood": -4.0, 
    "gabra": -3.0, "arrow": -2.5, "chao keng": -3.0, "cmi": -3.0, "gone case": -3.5, 
    "catch no ball": -2.0, "blur": -2.0, "blur sotong": -2.5, "kan cheong": -2.0,
    "siong": -2.5, "no standard": -2.5, "lousy": -3.0, "paiseh": -1.5, "mati": -3.0,

    # --- INTENSITY & EXCLAMATIONS ---
    "sibei": 2.0, "see peh": 2.0, "lagi": 1.5, "alamak": -1.5, "aiyoh": -1.0, 
    "aiyah": -1.0, "habis": -2.5, "wa lau": -1.0, "wa piang": -1.0, "kaypoh": -1.0,


    # Neutral/Contextual (0.0 Score) - Commonly used words that are culturally specific but don't inherently carry sentiment
    # Titles & People
    "abang": 0.0, "ah beng": 0.0, "ah lian": 0.0, "ah pek": 0.0, "ah soh": 0.0,
    "auntie": 0.0, "uncle": 0.0, "encik": 0.0, "makcik": 0.0, "bhai": 0.0,
    "mummy": 0.0, "ah chek": 0.0, "gu niang": 0.0, "nanny": 0.0,

    # Food & Drink (The 'Makan' List)
    "nasi lemak": 0.0, "prata": 0.0, "teh tarik": 0.0, "kopi": 0.0, "milo": 0.0,
    "bak kut teh": 0.0, "laksa": 0.0, "chicken rice": 0.0, "char kway teow": 0.0,
    "satay": 0.0, "roti john": 0.0, "chendol": 0.0, "ice kacang": 0.0, 
    "bandung": 0.0, "dim sum": 0.0, "kaya toast": 0.0, "beehoon": 0.0,
    "otah": 0.0, "popiah": 0.0, "thosai": 0.0, "murtabak": 0.0,

    # Places & Objects
    "void deck": 0.0, "mrt": 0.0, "kopitiam": 0.0, "pasar malam": 0.0,
    "nric": 0.0, "longkang": 0.0, "kampung": 0.0, "five-foot way": 0.0,
    "handphone": 0.0, "air-con": 0.0, "dhobi": 0.0, "hdb": 0.0,

    # Cultural & Misc
    "singapore": 0.0, "singlish": 0.0, "angpow": 0.0, "getai": 0.0,
    "halal": 0.0, "haram": 0.0, "bible": 0.0, "007": 0.0, "302": 0.0
}

    
    final_lexicon.update(full_slang)

    # 2. Multilingual Strategic Keywords
    pos_keywords = [
        "humanitarian", "thank you", "thankful", "grateful", "selfless", "hero", "proud", "support", "supportive", 
        "encourage", "inspire", "meaningful", "pride", "donate", "donation", "shared", "community", "initiative", 
        "motivate", "consistent", "kind", "fast", "short", "quick", "efficient", "helpful", "nice", "savior", 
        "saving", "save", "life", "success", "appreciate", "professional", "patient", "easy", "painless", "care", 
        "aid", "happy", "excited", "smooth", "excellent", "joy", "love", "gratitude", "positive", "resilient", 
        "compassion", "life-saving", "dedicated", "transparency", "accountable", "principled", "equitable", 
        "dignified", "timely", "exemplary", "compassionate", "strategic", "reliable", "uplifting",
        "人道", "谢谢", "感恩", "感激", "无私", "英雄", "自豪", "支持", "鼓励", "启发", "有意义", "捐赠", 
        "社区", "积极", "勤恳", "亲切", "快速", "高效", "帮助", "救星", "救命", "成功", "赞赏", "专业", 
        "耐心", "简单", "无痛", "照顾", "援助", "开心", "顺利", "优秀", "正能量", "坚韧", "同情心", 
        "敬业", "透明度", "负责", "有原则", "公平", "庄严", "及时", "典范", "慈悲", "战略性", "可靠", "振奋人心",
        "kemanusiaan", "terima kasih", "bersyukur", "hargai", "ikhlas", "wira", "bangga", "sokong", "galak", 
        "inspirasi", "bermakna", "derma", "komuniti", "inisiatif", "motivasi", "konsisten", "baik", "cepat", 
        "pantas", "cekap", "bantu", "penyelamat", "nyawa", "berjaya", "profesional", "sabar", "mudah", "selesa", 
        "jaga", "bantuan", "gembira", "lancar", "cemerlang", "tabah", "belas kasihan", "menyelamatkan nyawa", 
        "berdedikasi", "ketelusan", "bertanggungjawab", "berprinsip", "saksama", "bermaruah", "tepat pada masa", 
        "contoh terbaik", "ihsan", "strategik", "boleh dipercayai", "menyuntik semangat",
        "மனிதாபிமானம்", "நன்றி", "பாராட்டு", "வீரன்", "பெருமை", "ஆதரவு", "ஊக்கம்", "உந்துதல்", "அர்த்தமுள்ள", 
        "தானம்", "சமூக", "முயற்சி", "சீரான", "தயவு", "வேகம்", "திறமையான", "உதவி", "மீட்பர்", "வெற்றி", 
        "தொழில்முறை", "பொறுமை", "எளிதான", "வலியற்ற", "கவனிப்பு", "மகிழ்ச்சி", "சிறப்பு", "மீண்டெழும் திறன்", 
        "இரக்கம்", "உயிர் காக்கும்", "அர்ப்பணிப்பு", "வெளிப்படைத்தன்மை", "பொறுப்புள்ள", "கொள்கையுள்ள", "சமமான", 
        "கண்ணியமான", "சரியான நேரத்திற்கு", "முன்மாதிரியான", "கருணையுள்ள", "மூலோபாய", "நம்பகமான", "உற்சாகமளிக்கும்"
    ]

    neg_keywords = [
        "long queue", "long wait", "slow", "rude", "crowded", "painful", "scary", "pain", "bad", "issue", "problem", 
        "conflict", "messy", "disorganized", "disorganised", "unorganized", "unorganised", "waste", "poor", "upset", 
        "unhappy", "disappoint", "fail", "long", "impatient", "unprofessional", "hard", "difficult", "unreasonable", 
        "waste of time", "unresponsive", "ignored", "frustating", "frustrated", "confusing", "weird", "rough", 
        "aggressive", "hate", "disgust", "anger", "angry", "anxious", "anxiety", "stress", "mistake", "negative", 
        "ungrateful", "shortage", "urgent", "risky", "delayed", "misleading", "scandal", "ineffective", "exploitative", 
        "biased", "negligent", "misconduct", "discrepancy", "insecure", "exclusionary", "tardy",
        "排长龙", "等很久", "慢", "粗鲁", "拥挤", "痛苦", "恐怖", "痛", "坏", "问题", "冲突", "乱", "杂乱", 
        "浪费", "差", "伤心", "不开心", "失望", "失败", "久", "没耐心", "不专业", "难", "辛苦", "不合理", 
        "浪费时间", "没反应", "被无视", "烦躁", "困惑", "奇怪", "粗暴", "讨厌", "恶心", "生气", "焦虑", 
        "压力", "错误", "负面", "忘恩负义", "短缺", "紧急", "危险", "延误", "误导", "丑闻", "无效", 
        "剥削", "有偏见", "疏忽", "行为不检", "差异", "不安全", "排他性", "迟缓",
        "beratur panjang", "tunggu lama", "lambat", "biadab", "sesak", "sakit", "takut", "buruk", "masalah", 
        "konflik", "bersepah", "tidak teratur", "bazir", "lemah", "sedih", "kecewa", "gagal", "lama", 
        "tidak sabar", "tidak profesional", "susah", "sukar", "tidak munasabah", "buang masa", "tidak responsif", 
        "diabaikan", "geram", "bingung", "pelik", "kasar", "benci", "jijik", "marah", "cemas", "stres", "salah", 
        "negatif", "kekurangan", "segera", "berisiko", "tertangguh", "mengelirukan", "skandal", "tidak berkesan", 
        "eksploitasi", "berat sebelah", "cuai", "salah laku", "percanggahan", "tidak selamat", "memulau", "lewat",
        "நீண்ட வரிசை", "நீண்ட காத்திருப்பு", "மெதுவான", "முரட்டுத்தனமான", "நெரிசல்", "வலி", "பயம்", "மோசம்", 
        "பிரச்சனை", "மோதல்", "ஒழுங்கற்ற", "வீணான", "ஏமாற்றம்", "தோல்வி", "கடினமான", "பொறுமையற்ற", "பதிலளிக்காத", 
        "குழப்பம்", "வெறுப்பு", "கோபம்", "பதட்டம்", "மன அழுத்தம்", "தவறு", "எதிர்மறை", "பற்றாக்குறை", "அவசரம்", 
        "ஆபத்தானது", "தாமதம்", "தவறாக வழிநடத்தும்", "ஊழல்", "பயனற்ற", "சுரண்டல்", "சார்புடைய", "கவனக்குறைவான", 
        "தவறான நடத்தை", "முரண்பாடு", "பாதுகாப்பற்ற", "புறக்கணிக்கும்", "தாமதமான"
    ]

    neu_keywords = [
        "humanity", "criteria", "news", "report", "location", "address", "queue", "walk-in", "appointment", 
        "schedule", "information", "announcement", "announce", "eligible", "Level", "Bloodbank", "opening", 
        "travel", "O", "A", "B", "AB", "O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-", "partnership", 
        "collaboration", "history", "promo", "promotion", "giveaway", "come down", "come on down", "visit us", 
        "register here", "logistics", "deployment", "beneficiary",
        "人性", "标准", "新闻", "报告", "地点", "地址", "排队", "直接登门", "预约", "时间表", "信息", "公告", 
        "宣布", "合格", "楼层", "血库", "开放", "旅游", "合作伙伴", "合作", "历史", "促销", "抽奖", "亲临", 
        "参观", "注册", "物流", "部署", "受益者",
        "kriteria", "berita", "laporan", "lokasi", "alamat", "giliran", "janji temu", "jadual", "maklumat", 
        "pengumuman", "layak", "tingkat", "tabung darah", "pembukaan", "perjalanan", "kerjasama", "sejarah", 
        "promosi", "hadiah", "datanglah", "lawat", "daftar", "logistik", "kerahan", "penerima faedah",
        "செய்திகள்", "அறிக்கை", "இடம்", "முகவரி", "வரிசை", "முன்பதிவு", "அட்டவணை", "தகவல்", "அறிவிப்பு", 
        "தகுதி", "இரத்த வங்கி", "திறப்பு", "பயணம்", "கூட்டாண்மை", "வரலாறு", "விளம்பரம்", "வருகை தரவும்", 
        "பதிவு செய்யவும்", "தளவாடங்கள்", "பணியமர்த்தல்", "பயனாளி"
    ]

    # 3. Assemble and Calibrate
    for kw in pos_keywords: final_lexicon[kw.lower()] = positive_score
    for kw in neg_keywords: final_lexicon[kw.lower()] = negative_score
    for kw in neu_keywords: final_lexicon[kw.lower()] = neutral_score

    analyzer.lexicon.update(final_lexicon)

    print("✅ BLOCK 1: SETUP SUCCESSFUL")
    print(f"   - Sentiment Lexicon size: {len(analyzer.lexicon)} words")

except ImportError as e:
    print(f"❌ BLOCK 1: FAIL - Missing Library: {e}")
except Exception as e:
    print(f"❌ BLOCK 1: FAIL - Error at line {traceback.extract_stack()[-1].lineno}: {e}")

✅ BLOCK 1: SETUP SUCCESSFUL
   - Sentiment Lexicon size: 8067 words


### **Block 2: Category Keyword Dictionary**
This block defines the **Keyword Engine** used to categorize news articles. It is structured to identify strategic terms across Singapore’s four official languages to ensure comprehensive media monitoring.

**Multilingual Coverage:** Includes specific keyword lists for **English (EN), Chinese (ZH), Malay (MS), and Tamil (TA).**

**Strategic Buckets:** News is automatically sorted into Company News (Direct SRC mentions), Industry News (Social service/Youth sector), or Partner News (Collaborative humanitarian entities).
<br>


<br>

**Key Components:**

**Organization & Services:** Monitors specific SRC entities like "Red Cross Home for the Disabled" or the "Bloodbank@HSA".

**Social Sector Monitoring:** Tracks broader trends in "philanthropy", "organ transplant", and "elder care".

**Partner Tracking:** Keeps tabs on collaborative efforts with entities such as "Habitat for Humanity" and "Mercy Relief".

In [3]:
# --- BLOCK 2: CATEGORY KEYWORD DICTIONARY (FULL RED CROSS LIST) ---

CATEGORY_KEYWORDS = {
    "company_news": {
        "en": [
            "International Red Cross", "International Committee of the Red Cross", "ICRC", 
            "International Federation of Red Cross", "IFRC", "Red Cross Society", 
            "Singapore Red Cross Society", "Singapore Red Cross", "SRC", "organization", 
            "(SRC)", "NGO", "organisation", "Tan Kai Hoe", "Singapore Red Cross Academy", 
            "Red Cross Home for the Disabled", "Blood Donor Recruitment Programme", 
            "national blood programme", "Blood Donor Programme", "Blood Buddy", 
            "Bloodbank@HSA", "Bloodbank@One Punggol", "Bloodbank@Dhoby Ghaut", 
            "Bloodbank@Woodlands", "Bloodbank@Westgate Tower", "Bloodbank | Westgate", 
            "Bloodmobile drive", "Blood drive", "transfuse blood", "national blood stock", 
            "national blood stocks", "Red Cross Ambulance Service", "red cross emergency ambulance", 
            "First Aid", "Firstaid", "Cardiopulmonary resuscitation", "Psychological first aid", 
            "Clinic on Wheels", "Community First Aid", "Community FirstAid", "Red Cross Youth", 
            "red cross junior", "Red Cross Junior @ Community", "foodaid", "TransportAid", 
            "ElderAid", "Elder Aid", "Home Monitoring and Eldercare", "International Bazaar", 
            "Community-Led Action for Resilience", "CommunityLed Action for Resilience", 
            "Community Led Action for Resilience"
        ],
        "zh": [
            "国际红十字会", "红十字国际委员会", "ICRC", "红十字会与红新月会国际联合会", 
            "IFRC", "红十字会", "新加坡红十字会", "新加坡红十字会 (SRC)", "机构", 
            "非政府组织", "非政府组织(NGO)", "陈开河", "新加坡红十字会学院", 
            "红十字会残疾人士之家", "红十字会残障人士之家", "捐血者招募计划", 
            "献血者招募计划", "国家血液计划", "捐血者计划", "捐血伙伴", "Blood Buddy", 
            "HSA 血库", "榜鹅坊血库", "多美歌血库", "兀兰血库", "西城大厦血库", "西城 (Westgate)", 
            "移动捐血活动", "捐血活动", "输血", "国家血液储备", "红十字会救护车服务", 
            "红十字会紧急救护车", "急救", "FirstAid", "心肺复苏术", "(CPR)", "心理急救", 
            "移动诊所", "社区急救", "红十字青年", "红十字青年团", "红十字少年组", 
            "社区红十字少年组", "食物援助", "FoodAid", "交通援助", "TransportAid", 
            "乐龄援助", "ElderAid", "Elder Aid", "居家监控与乐龄护理", "国际慈善义卖", 
            "社区主导的抗韧性行动", "Community-Led Action for Resilience"
        ],
        "ms": [
            "Palang Merah Antarabangsa", "Jawatankuasa Palang Merah Antarabangsa", "ICRC", 
            "Persekutuan Antarabangsa Persatuan Palang Merah", "IFRC", "Palang Merah", 
            "Persatuan Palang Merah Singapura", "Persatuan Palang Merah Singapura Society", 
            "organisasi", "Pertubuhan Bukan Kerajaan (NGO)", "Tan Kai Hoe", 
            "Akademi Palang Merah Singapura", "Rumah Palang Merah untuk Orang Kurang Upaya", 
            "Program Pengambilan Penderma Darah", "program darah kebangsaan", 
            "Program Penderma Darah", "Blood Buddy", "Tabung Darah@HSA", "Tabung Darah@One Punggol", 
            "Tabung Darah@Dhoby Ghaut", "Tabung Darah@Woodlands", "Tabung Darah@Westgate Tower", 
            "Kempen Derma Darah Bergerak", "Kempen Derma Darah", "pemindahan darah", 
            "stok darah kebangsaan", "Perkhidmatan Ambulans Palang Merah", 
            "ambulans kecemasan palang merah", "Pertolongan Cemas", "Resusitasi Kardiopulmonari (CPR)", 
            "Pertolongan Cemas Psikologi", "Klinik Bergerak", "Pertolongan Cemas Komuniti", 
            "Belia Palang Merah", "junior palang merah", "Junior Palang Merah @ Komuniti", 
            "Bantuan Makanan", "Bantuan Pengangkutan", "Bantuan Warga Emas", 
            "Pemantauan Rumah and Penjagaan Warga Emas", "Bazar Antarabangsa", 
            "Tindakan Terajui Komuniti untuk Ketahanan"
        ],
        "ta": [
            "பன்னாட்டுச் செஞ்சிலுவை சங்கம்", "பன்னாட்டுச் செஞ்சிலுவைச் சங்கக் குழு", 
            "பன்னாட்டுச் செஞ்சிலுவை சங்க கூட்டமைப்பு", "செஞ்சிலுவை சங்கம்", 
            "சிங்கப்பூர் செஞ்சிலுவை சங்கம்", "அமைப்பு", "நிறுவனம்", 
            "தன்னார்வத் தொண்டு நிறுவனம்", "டான் கை ஹோ", "சிங்கப்பூர் செஞ்சிலுவை அகாதமி", 
            "மாற்றுத்திறனாளிகளுக்கான செஞ்சிலுவை சங்கம்", "இரத்தக் கொடையாளர் சேர்த்தல் திட்டம்", 
            "தேசிய இரத்தத் திட்டம்", "இரத்தக் கொடையாளர் திட்டம்", "இரத்த தானத் தோழன்", 
            "இரத்த தானத் தோழி", "HSA இரத்த வங்கி", "ஒன் பொங்கோல் இரத்த வங்கி", 
            "டோபி காட் இரத்த வங்கி", "வுட்லண்ட்ஸ் இரத்த வங்கி", "வெஸ்ட்கேட் டவர் இரத்த வங்கி", 
            "நடமாடும் இரத்த தான ஊர்தி", "தேசிய இரத்த இருப்பு", "இரத்தமேற்றுதல்", 
            "செஞ்சிலுவைச் சங்க ஆம்புலன்ஸ் சேவை", "செஞ்சிலுவைச் சங்க அவசர ஆம்புலன்ஸ் சேவை", 
            "முதலுதவி", "முதலுதவி சிகிச்சை", "இதய நுரையீரல் புத்துயிர்ப்பு", 
            "உளவியல் முதலுதவி", "நடமாடும் மருத்துவமனை", "சமூக முதலுதவி", 
            "செஞ்சிலுவை இளையர் பிரிவு", "செஞ்சிலுவைச் சங்க இளையர்", 
            "செஞ்சிலுவைச் சங்க இளையோர்", "சமூகச் செஞ்சிலுவை இளையர்", "உணவு உதவி", 
            "போக்குவரத்து உதவி", "முதியோர் உதவி", "வீட்டு கண்காணிப்பு மற்றும் முதியோர் பராமரிப்பு", 
            "தாங்குதிறனுக்கான சமூகத் தலைமை நடவடிக்கை"
        ]
    },
    "industry_news": {
        "en": [
            "National Youth Council", "National Youth Survey", "YouthSCOPE", "Singapore Youth Award", 
            "Youth Corp", "Singapore Youth", "Y+", "youth", "YOUTH.sg", "Youth Statistics in Brief", 
            "State of Youth", "community", "Programmes", "portal", "development", "engagement", 
            "leadership", "Youth Development", "Engagement", "Youth Council", "Young ChangeMakers",
            "organ transplant", "rejection", "advancements", "issues", "experience", "denial", 
            "elder care aging population", "donate blood", "donors", "blood", "care & share", 
            "com care", "grant making", "grant-making", "com", "social service", "donation", 
            "s'pore", "singapore", "social services", "foundation", "foundations", "national council",
            "social", "fund", "funded", "sg united", "giving", "fundraising", "volunteer", 
            "volunteering", "philanthropy", "charitable", "charities", "grantmaking"
        ],
        "zh": [
            "全国青年理事会", "全国青年调查", "YouthSCOPE", "新加坡青年奖", "青年志愿军", "新加坡", 
            "青年志愿军", "Y+", "青年", "YOUTH.sg", "青年统计简报", "青年现状", "计划", "社区", 
            "门户网站", "发展", "参与", "领导力", "青年发展", "青年理事会", "青年变革者", 
            "器官移植", "排斥", "进展", "技术进展", "课题", "经验", "拒绝", "捐赠", "医疗机构", 
            "紧急", "捐血", "输血", "乐龄护理与人口老龄化", "关怀与分享", "社区关怀计划", "ComCare", 
            "拨发补助金", "社保援助", "社会服务", "捐助", "基金会", "全国理事会", "资助", "新加坡联合", 
            "给予", "筹款", "志愿者", "义工", "慈善", "慈善机构", "拨发助学金"
        ],
        "ms": [
            "Majlis Belia Kebangsaan", "Kajian Belia Kebangsaan", "YouthSCOPE", "Anugerah Belia Singapura", 
            "Youth Corp", "Singapura", "belia", "YOUTH.sg", "Ringkasan Statistik Belia", "Keadaan Belia", 
            "komuniti", "Program", "portal", "pembangunan", "penglibatan", "kepimpinan", 
            "Pembangunan Belia", "Young ChangeMakers", "pemindahan organ", "penolakan", "kemajuan", 
            "isu", "pengalaman", "penafian", "penjagaan warga emas", "khidmat sosial", "derma", 
            "penyaluran geran", "penyaluran", "yayasan", "majlis kebangsaan", "sosial", "dana", 
            "dibiayai", "sg bersatu", "pemberian", "pengumpulan dana", "sukarelawan", "dermawan", "kebajikan"
        ],
        "ta": [
            "தேசிய இளைஞர் மன்றம்", "தேசிய இளைஞர் ஆய்வு", "YouthSCOPE", "சிங்கப்பூர் இளைஞர் விருது", 
            "சிங்கப்பூர் இளைஞர் படை", "Y+", "YOUTH.sg", "இளைஞர் புள்ளிவிவரச் சுருக்கம்", 
            "சிங்கப்பூர் இளைஞரின் நிலை", "இளைஞர் சமூகம்", "இளைஞர் திட்டங்கள்", "இளைஞர் இணையவாசல்", 
            "இளைஞர் மேம்பாடு", "இளைஞர் ஈடுபாடு", "இளைஞர் தலைமைத்துவம்", "இளம் மாற்றத்தை உருவாக்குபவர்கள்", 
            "இளைஞர் மன்றம்", "உறுப்பு மாற்று அறுவை சிகிச்சை", "உறுப்பு மாற்று நிராகரிப்பு", 
            "மாற்றத்தில் புதிய முன்னேற்றங்கள்", "உறுப்பு மாற்றுச் சிக்கல்கள்", "உறுப்பு மாற்று அறுவை சிகிச்சை அனுபவம்", 
            "உறுப்பு மாற்று மறுப்பு", "முதியோர் மக்கள் தொகை", "பகிர்ந்து கொள்வோம் மற்றும் பராமரிப்போம்", 
            "Care & Share", "காம்கேர்", "நிதியுதவி", "மானியங்கள்", "அறக்கட்டளை", "நன்கொடை", "சமூக சேவைத் துறை", 
            "தேசிய சமூக சேவை மன்றம்", "சிங்கப்பூர்", "சமூகச் சேவை", "நிதியளிக்கப்பட்டது", "எஸ்ஜி யுனைடெட்", 
            "ஈகை", "நிதி திரட்டுதல்", "தன்னார்வலர்", "தன்னார்வத் தொண்டு", "பரோபகாரம்", "தருமம்", "தொண்டு நிறுவனங்கள்"
        ]
    },
    "partner_news": {
        "en": [
            "Habitat for Humanity", "International Community of the Red Cross (ICRC)", "ICRC",
            "International Federation of Red Cross and Red Crescent Societies (IFRC)", "IFRC",
            "Mercy Relief", "Singapore International Foundation", "SIF", "Our Better World",
            "SIF Connect", "The Red Pencil Singapore", "The Red Pencil (Singapore)"
        ],
        "zh": [
            "仁人家园", "红十字国际委员会", "ICRC", "红十字会与红新月会国际联合会", "IFRC",
            "慈援组织", "组织慈援", "Mercy Relief", "新加坡国际基金会", "企业新闻", 
            "我们的美好世界", "Our Better World", "新加坡国际基金会交流", "SIF Connect",
            "新加坡红铅笔", "The Red Pencil (新加坡)"
        ],
        "ms": [
            "Habitat for Humanity", "Jawatankuasa Palang Merah Antarabangsa", "ICRC",
            "Persekutuan Antarabangsa Persatuan Palang Merah dan Bulan Sabit Merah", "IFRC",
            "Mercy Relief", "SIF - Berita Korporat", "Yayasan Antarabangsa Singapura",
            "Our Better World", "SIF Connect", "The Red Pencil Singapura"
        ],
        "ta": [
            "ஹபிடாட் ஃபார் ஹியூமானிட்டி", "மனிதநேயத்திற்கான வாழ்விடம்", "வடிவமைப்புத் தொண்டு",
            "பன்னாட்டுச் செஞ்சிலுவைச் சங்கக் குழு", "சர்வதேச செஞ்சிலுவைச் சங்கக் குழு", "ஐசிஆர்சி",
            "சர்வதேச செஞ்சிலுவை மற்றும் செம்பிறை சங்கங்களின் கூட்டமைப்பு", "ஐஎஃப்ஆர்சி",
            "மெர்சி ரிலீஃப்", "கருணை நிவாரணம்", "சிங்கப்பூர் சர்வதேச அறக்கட்டளை", "எமது சிறந்த உலகம்",
            "எஸ்ஐஎஃப் கனெக்ட்", "தி ரெட் பென்சில் சிங்கப்பூர்"
        ]
    }
}

### **Block 3: Logic Functions**
This block contains the core functions that categorize and score your data. It is designed to be language-aware and resilient to errors.

**Language-Aware Processing:** The script detects the language based on the matched keyword to determine if translation is required.

**Resilient Sentiment Scoring:** Combines translation logic with the calibrated VADER analyzer to handle multi-language data without crashing.<br>
<br>

**Key Components**<br>

**classify_article:** Standardizes text and matches keywords from your dictionary to assign a Category and Language.

**get_sentiment:** Bridges the language gap. It translates non-English comments before scoring them with our calibrated VADER analyzer (969 emojis + local slang).

**Safety Logic:** Uses try-except blocks to prevent the script from crashing if a specific translation fails or a row is empty.

In [4]:
# --- BLOCK 3: LOGIC FUNCTIONS (Classification & Sentiment) ---

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Safety check for googletrans installation
try:
    from googletrans import Translator
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "googletrans==4.0.0-rc1"])
    from googletrans import Translator

# 1. INITIALIZE TOOLS
analyzer = SentimentIntensityAnalyzer()
translator = Translator()

# 2. UPDATE VADER BRAIN WITH EMOJI DATASET
try:
    emoji_df = pd.read_csv("Emoji_Sentiment_Data_v1.0.csv")
    new_emoji_lexicon = {}
    
    for _, row in emoji_df.iterrows():
        try:
            # Convert Hex (0x1f602) to actual emoji (😂)
            hex_code = str(row['Unicode codepoint']).replace('0x', '').strip()
            emoji_icon = chr(int(hex_code, 16))
            
            # Calculate score from your CSV columns: (Pos - Neg) / Total
            pos = float(row['Positive'])
            neu = float(row['Neutral'])
            neg = float(row['Negative'])
            total = pos + neu + neg
            
            if total > 0:
                # Normalize to -1 to 1, then scale to VADER's -4 to 4 scale
                raw_score = (pos - neg) / total
                new_emoji_lexicon[emoji_icon] = raw_score * 4
        except:
            continue
            
    analyzer.lexicon.update(new_emoji_lexicon)
    print(f"✅ Successfully loaded {len(new_emoji_lexicon)} emojis into VADER.")
except FileNotFoundError:
    print("❌ Error: Emoji_Sentiment_Data_v1.0.csv not found in current folder.")

# 3. ADD LOCAL OVERRIDES (Singapore Red Cross Context)
local_context = {
    "🩸": 1.5, "🇸🇬": 2.0, "💪": 2.5, "💉": 2.0,   # Donation act
    "🎁": 1.5,   # Gift of life
    "🦁": 2.0,   # SG Pride
    "🙌": 2.0,   # Success/Celebration
    "❤️‍🩹": 2.0,  # Healing/Recovery
    "📢": 1.0,   # Important announcement
    "🏥": 1.0,   # Healthcare/Clinic support
    
}
analyzer.lexicon.update(local_context)

# --- LOGIC FUNCTIONS ---

def classify_article(text: str) -> dict:
    """Standardizes text and scans for keywords across categories."""
    clean_text = str(text).lower().strip()
    for category, langs in CATEGORY_KEYWORDS.items():
        for lang, keywords in langs.items():
            if any(kw.lower() in clean_text for kw in keywords):
                return {"keyword_category": category, "language": lang}
    return {"keyword_category": "other", "language": "en"}

def get_sentiment(text: str, lang: str) -> float:
    """Translates non-English text, then scores with updated VADER."""
    clean_text = str(text).strip()
    if not clean_text:
        return 0.0
    
    # Translate if necessary
    if lang in ['ta', 'zh', 'ms']:
        try:
            # We use .text to extract the string from the translation object
            translated = translator.translate(clean_text, dest='en').text
            return analyzer.polarity_scores(translated)['compound']
        except:
            return 0.0
            
    # Score English or Translated text directly
    return analyzer.polarity_scores(clean_text)['compound']

✅ Successfully loaded 969 emojis into VADER.


### **Block 4: The Pipeline**
This final block runs the logic from Block 3 on your cleaned dataset, generates the results, and saves everything into a master report (SRC_Sentiment_Final_Report.json/csv).

**Data-Wide Application:** The script iterates through all 272 rows of your unified_cleaned.csv, ensuring every comment is processed through the Singaporean-calibrated "Brain."

**Automated Categorization:** Beyond raw scores, the script assigns human-readable labels (Positive, Neutral, Negative) to make the data ready for high-level management presentations.<br>
<br>

**Key Components** <br>

**Score Generation:** Applies the get_sentiment function to create the sentiment_score column, factoring in the 600+ local terms and emojis.Sentiment Labeling:<br> <br>
Categorizes scores into human-readable labels:Positive, Negative, Neutral Scores between them. Final CSV Output: Exports the updated dataframe as SRC_Sentiment_Final_Report.csv. 

In [5]:
# --- BLOCK 4: EXECUTION ---

# 1. Load your cleaned data
df = pd.read_csv("unified_cleaned.csv")

# Ensure all comments are strings and handle missing values
df['comment_text'] = df['comment_text'].fillna('').astype(str)

print(f"Processing {len(df)} rows 🚀... This may take a moment.")

# 2. Apply classification and detect language
# We use the functions defined in Block 3
print("Categorizing comments and detecting languages...")
classification_results = df['comment_text'].apply(classify_article)
df['keyword_category'] = classification_results.apply(lambda x: x['keyword_category'])
df['language'] = classification_results.apply(lambda x: x['language'])

# 3. Apply the "Smart" Sentiment Analysis
# We use the analyzer that now knows your 969 emojis and custom terms!
print("Calculating sentiment scores (including translations)...")
df['sentiment_score'] = df.apply(lambda row: get_sentiment(row['comment_text'], row['language']), axis=1)

# 4. Create readable labels
def create_label(score):
    if score >= 0.05: return 'Positive'
    if score <= -0.05: return 'Negative'
    return 'Neutral'

df['sentiment_label'] = df['sentiment_score'].apply(create_label)

# 5. Save the final "Master File"
df.to_csv("SRC_Sentiment_Final_Report.csv", index=False)

print("\n" + "="*30)
print("✅ DONE! File saved as: SRC_Sentiment_Final_Report.csv")
print("="*30)

# 6. Show a quick preview of results
print("\n--- Sample of Processed Data ---")
print(df[['comment_text', 'keyword_category', 'sentiment_label', 'sentiment_score']].head(10))

Processing 272 rows 🚀... This may take a moment.
Categorizing comments and detecting languages...
Calculating sentiment scores (including translations)...

✅ DONE! File saved as: SRC_Sentiment_Final_Report.csv

--- Sample of Processed Data ---
                                        comment_text keyword_category  \
0  I think first aid and CPR needs a revamp. You ...     company_news   
1  Never do this.\n\nFirst thing you learn in fir...     company_news   
2                            First aid❌️\nLast aid✅️     company_news   
3  Imagine getting stabbed and your paramedic sta...     company_news   
4  Nah if i get stabbed and my paramedic be smili...     company_news   
5                          First aid 👎\nFirst dead 👍     company_news   
6                        Ok i got my First aid bag 😂     company_news   
7  I hate seeing stabbing first aid videos where ...     company_news   
8  Sollte in Berlin zur Pflicht werden solch ein ...     company_news   
9  So no first aid. Just p

In [6]:
import pandas as pd
import random

# Load the data
df_final = pd.read_csv("SRC_Sentiment_Final_Report.csv")

# Pick a random sample of 20-30 rows
sample_size = 20
sample = df_final.sample(n=sample_size).copy()

correct_count = 0

print(f"--- 🎯 ACCURACY CHECKER ({sample_size} Samples) ---")
print("Instructions: Read the comment and the AI label. Type 'y' if you agree, 'n' if it's wrong.")

for i, row in sample.iterrows():
    print(f"\nComment: {row['comment_text']}")
    print(f"AI Predicted: {row['sentiment_label']} (Score: {row['sentiment_score']})")
    
    user_input = input("Is the AI correct? (y/n): ").lower()
    if user_input == 'y':
        correct_count += 1

# Calculate Percentage
accuracy_pct = (correct_count / sample_size) * 100
print(f"\n" + "="*30)
print(f"✅ YOUR MODEL ACCURACY: {accuracy_pct}%")
print(f"="*30)

--- 🎯 ACCURACY CHECKER (20 Samples) ---
Instructions: Read the comment and the AI label. Type 'y' if you agree, 'n' if it's wrong.

Comment: Thank you for all..i have been looking for someone who teaches me to perform first aid simply until i found your channel🤍
AI Predicted: Positive (Score: 0.3612)

Comment: The Red Cross Youth March is a song used in the Red Cross Youth movement, which promotes morality, ethics, and social ...
AI Predicted: Positive (Score: 0.34)

Comment: its a first aid on a stsb wound
AI Predicted: Neutral (Score: 0.0)

Comment: Learn the procedure for performing CPR on an unresponsive person in this video from St John Ambulance. CPR stands for ...
AI Predicted: Neutral (Score: 0.0)

Comment: ILIGAN CITY RED CROSS YOUTH.
AI Predicted: Neutral (Score: 0.0)

Comment: For more information visit http://www.sja.org.uk St John Ambulance on Facebook - http://www.facebook.com/sja St John ...
AI Predicted: Neutral (Score: 0.0)

Comment: Learn First Aid and CPR. Become a L

In [7]:
import json
df_renamed = df.rename(columns={
    'text': 'title',
    'date': 'date_published',
    'user': 'author',
    'platform': 'source',
    'category': 'keyword_category',
    'lang': 'language',
    'sentiment': 'sentiment_score'
})

# 2. Convert to the list-of-dictionaries format
json_output = df_renamed.to_dict(orient='records')

# 3. Save to file
with open('final_report.json', 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=4)

print("File 'final_report.json' created successfully in the target format.")

File 'final_report.json' created successfully in the target format.
